# PlaSim emulator — slide figures

Emulator vs the SFNO **v11** baseline, held-out PlaSim sim52 years 122–131.

**Every line on every plot is one line of code.** Comment a line out to drop that
series; copy a line to add one. Nothing is hidden in a loop.

```python
fig, ax = panel("2 m temperature — global", "RMSE (K)")
add(ax, "v11",    "tas_rmse_global")
add(ax, "stage4", "tas_rmse_global")
# add(ax, "stage1", "tas_rmse_global")   <- commented out: not drawn
add(ax, "r2",     "tas_rmse_global")
finish(ax); save(fig, "my_figure")
```

One figure per cell, one panel per figure — so each drops straight onto its own slide.

**Kernel:** the `aires` conda env (has matplotlib; needs neither torch nor makani):

```
/work2/11079/aasch/stampede3/conda-envs/aires/bin/python -m ipykernel install \
    --user --name aires --display-name "Python (aires)"
```

**Colours:** Okabe–Ito, the standard colour-vision-deficiency-safe palette (worst
adjacent ΔE 9.6 deuteranopia, 20.0 normal vision). Each series also has its own line
style, so the figures survive greyscale printing and any form of colour blindness.

In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

RES = "/work2/11079/aasch/stampede3/runs/plasim_pnemo"
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

# ---------------- data ----------------
_d = json.load(open(f"{RES}/skill_comparison.json"))
_d["models"]["r2"] = json.load(open(f"{RES}/skill_r2_only.json"))["models"]["r2"]
LEAD = np.array(_d["lead_hours"]) / 24.0        # 6-hourly, to 15 days
M    = _d["models"]

DIAG = {"stage4": json.load(open(f"{RES}/diagnose_stage4_cpu.json")),
        "r2":     json.load(open(f"{RES}/diagnose_r2_unroll32.json"))}
SPREAD = json.load(open(f"{RES}/spread_diag.json"))
TAIL   = json.load(open(f"{RES}/daily_tail_compare.json"))
DAYS   = np.array(DIAG["stage4"]["days"])

CLIM_PRECIP, CLIM_T2M = 5.108, 7.006   # measured no-skill climatology baselines

# ---------------- style ----------------
# Okabe-Ito + a distinct dash pattern per series (second encoding for CVD/greyscale)
STYLE = {
 "v11":    dict(color="#000000", ls=(0, (5, 3)),       label="SFNO v11 (baseline)"),
 "stage1": dict(color="#56B4E9", ls=(0, (1, 1.5)),     label="R1 s1 · single-step"),
 "stage2": dict(color="#E69F00", ls=(0, (3, 1, 1, 1)), label="R1 s2 · unroll 2"),
 "stage3": dict(color="#CC79A7", ls=(0, (4, 1.5)),     label="R1 s3 · unroll 4"),
 "stage4": dict(color="#D55E00", ls="-",               label="R1 s4 · ensemble"),
 "r2":     dict(color="#0072B2", ls="-",               label="R2 · rollout to 8 d"),
 "obs":    dict(color="#555555", ls=(0, (1, 2)),       label="observed"),
}

plt.rcParams.update({
 "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
 "font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
 "legend.fontsize": 12, "xtick.labelsize": 13, "ytick.labelsize": 13,
 "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.7,
 "axes.spines.top": False, "axes.spines.right": False,
 "lines.linewidth": 2.6, "legend.frameon": False,
 "figure.facecolor": "white", "axes.facecolor": "white",
})

# ---------------- one-line-per-series helpers ----------------
def panel(title, ylabel, xlabel="lead time (days)", figsize=(8.4, 5.3)):
    """Start a figure. Returns (fig, ax)."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel(xlabel)
    return fig, ax

def add(ax, model, metric, **kw):
    """One series from the 6-hourly lead-time results. `kw` overrides label/color/ls."""
    st = {**STYLE[model], **kw}
    return ax.plot(LEAD, M[model][metric], **st)[0]

def add_skill(ax, model, metric, clim, **kw):
    """Same, expressed as % skill vs a climatology baseline (0 = no skill)."""
    st = {**STYLE[model], **kw}
    return ax.plot(LEAD, 100 * (1 - np.array(M[model][metric]) / clim), **st)[0]

def add_daily(ax, model, key, **kw):
    """One series from the daily-mean diagnostics: crps_pred, crps_true, csi, crps_wet, pod, far."""
    st = {**STYLE[model], **kw}
    return ax.plot(DAYS, DIAG[model][key], **st)[0]

def add_spread(ax, model, key, **kw):
    """One series from the corrected single-member diagnostics:
    member_spec, mean_spec, q99_member, q99_pooled, q99_obs, spread_skill."""
    st = {**STYLE[model], **kw}
    return ax.plot(DAYS, SPREAD[model][key], **st)[0]

def add_ref(ax, y, label=None, **kw):
    """Horizontal reference line (climatology, 1.0, ...)."""
    st = dict(color="#555555", lw=1.5, ls=":", zorder=0); st.update(kw)
    ln = ax.axhline(y, **st)
    if label:
        ax.annotate(label, xy=(0.012, y), xycoords=("axes fraction", "data"),
                    va="bottom", fontsize=11, color="#555555")
    return ln

def finish(ax, xmax=None, legend="best", xtick=3):
    if xmax: ax.set_xlim(0, xmax)
    ax.xaxis.set_major_locator(MultipleLocator(xtick))
    if legend: ax.legend(loc=legend)
    return ax

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT}/{name}.{ext}")
    print(f"saved {OUT}/{name}.png + .pdf")

print("ready — models:", list(M), "| leads to", LEAD[-1], "d | daily to", DAYS[-1], "d")

## What you can plot

| helper | data | keys |
|---|---|---|
| `add` | 6-hourly, to 15 d | `tas_rmse_global`, `tas_rmse_Chicago`, `tas_crps_global`, `tas_crps_Chicago`, `pr_rmse_global`, `pr_rmse_Bay_of_Bengal`, `pr_crps_global`, `pr_crps_Bay_of_Bengal` |
| `add_skill` | same, as % vs climatology | any of the above + `CLIM_T2M` / `CLIM_PRECIP` |
| `add_daily` | daily mean, to 10 d | `crps_pred`, `crps_true`, `csi`, `pod`, `far`, `crps_wet` |
| `add_spread` | single members, to 10 d | `member_spec`, `q99_member`, `q99_obs`, `spread_skill`, `mean_spec`, `q99_pooled` |

Models: `v11`, `stage1`, `stage2`, `stage3`, `stage4`, `r2` — `add_daily`/`add_spread`
have `stage4` and `r2` only.

## Fig 1 — 2 m temperature, global

*Round 2 beats the v11 baseline at every lead.* The clean comparison: `tas` is a
prognostic state channel and v11 shares our units and climatology, so nothing is rescaled.

In [ ]:
fig, ax = panel("2 m temperature — global", "RMSE (K)")
add(ax, "v11",    "tas_rmse_global")
add(ax, "stage1", "tas_rmse_global")
add(ax, "stage2", "tas_rmse_global")
add(ax, "stage3", "tas_rmse_global")
add(ax, "stage4", "tas_rmse_global")
add(ax, "r2",     "tas_rmse_global")
add_ref(ax, CLIM_T2M, "climatology (no skill)")
finish(ax, xmax=15, legend="upper left")
save(fig, "fig1_t2m_global"); plt.show()

## Fig 2 — 2 m temperature, Chicago

Same metric over the 3×3 Chicago stencil. Trimmed to the three lines that matter for a
talk; uncomment the others if your advisor wants the full stage progression.

In [ ]:
fig, ax = panel("2 m temperature — Chicago", "RMSE (K)")
add(ax, "v11",    "tas_rmse_Chicago")
# add(ax, "stage1", "tas_rmse_Chicago")
# add(ax, "stage2", "tas_rmse_Chicago")
# add(ax, "stage3", "tas_rmse_Chicago")
add(ax, "stage4", "tas_rmse_Chicago")
add(ax, "r2",     "tas_rmse_Chicago")
finish(ax, xmax=15, legend="upper left")
save(fig, "fig2_t2m_chicago"); plt.show()

## Fig 3 — precipitation, global

**Caveat for the slide:** v11's precipitation channel is *not* our field in different
units — mean ratio 3496 vs std ratio 4331, zero fraction 12% against our 31%, because its
postprocessing interpolates. It is **z-score matched** onto our distribution first, so its
line is **pattern skill, not absolute error**.

In [ ]:
fig, ax = panel("Precipitation — global", "RMSE (mm/day)")
add(ax, "v11",    "pr_rmse_global")
add(ax, "stage1", "pr_rmse_global")
add(ax, "stage2", "pr_rmse_global")
add(ax, "stage3", "pr_rmse_global")
add(ax, "stage4", "pr_rmse_global")
add(ax, "r2",     "pr_rmse_global")
add_ref(ax, CLIM_PRECIP, "climatology (no skill)")
finish(ax, xmax=15, legend="lower right")
save(fig, "fig3_precip_global"); plt.show()

## Fig 4 — precipitation, Bay of Bengal

The 3×3 stencil AI-RES scores on.

In [ ]:
fig, ax = panel("Precipitation — Bay of Bengal", "RMSE (mm/day)")
add(ax, "v11",    "pr_rmse_Bay_of_Bengal")
# add(ax, "stage1", "pr_rmse_Bay_of_Bengal")
# add(ax, "stage2", "pr_rmse_Bay_of_Bengal")
add(ax, "stage3", "pr_rmse_Bay_of_Bengal")
add(ax, "stage4", "pr_rmse_Bay_of_Bengal")
add(ax, "r2",     "pr_rmse_Bay_of_Bengal")
finish(ax, xmax=15, legend="lower right")
save(fig, "fig4_precip_bob"); plt.show()

## Fig 5 — skill vs climatology, and the no-skill crossover

Likely the strongest single slide. Zero is where a model stops beating a plain
climatological forecast. **Round 2 pushes the crossover from v11's ~day 9.3 to ~day 12.6.**

In [ ]:
fig, ax = panel("Global precipitation skill", "skill vs climatology (%)")
add_skill(ax, "v11",    "pr_rmse_global", CLIM_PRECIP)
add_skill(ax, "stage1", "pr_rmse_global", CLIM_PRECIP)
add_skill(ax, "stage2", "pr_rmse_global", CLIM_PRECIP)
add_skill(ax, "stage3", "pr_rmse_global", CLIM_PRECIP)
add_skill(ax, "stage4", "pr_rmse_global", CLIM_PRECIP)
add_skill(ax, "r2",     "pr_rmse_global", CLIM_PRECIP)
add_ref(ax, 0, color="#222222", lw=1.6, ls="-")
ax.set_ylim(-60, 80)
finish(ax, xmax=15, legend="upper right")
save(fig, "fig5_precip_skill_score"); plt.show()

for k in ["v11", "stage4", "r2"]:
    s = 1 - np.array(M[k]["pr_rmse_global"]) / CLIM_PRECIP
    b = np.where(s <= 0)[0]
    print(f"{k:7s} crossover: {'beyond 15 d' if not len(b) else f'{LEAD[b[0]]:.1f} d'}")

## Fig 6 — daily-mean CRPS, and *why* skill decays

The **dotted** lines feed the model the true state at each lead instead of its own
forecast. They are flat — so given a perfect state the precipitation head is excellent and
stays excellent at every lead. All of the decay is state error propagating into the
diagnosis.

*Slide caption: the precipitation head was never the bottleneck — the state is.*

In [ ]:
fig, ax = panel("Daily-mean precipitation CRPS", "CRPS (mm/day)")
add_daily(ax, "stage4", "crps_pred")
add_daily(ax, "r2",     "crps_pred")
add_daily(ax, "stage4", "crps_true", ls=(0, (2, 2)), lw=2.0, label="stage 4 — TRUE state")
add_daily(ax, "r2",     "crps_true", ls=(0, (2, 2)), lw=2.0, label="round 2 — TRUE state")
finish(ax, xmax=10, legend="upper left", xtick=2)
save(fig, "fig6_daily_crps"); plt.show()

## Fig 7 — small-scale power in a *single* member

Measured on single members, not the ensemble mean. That distinction matters: an earlier
version of this analysis compared the ensemble *mean* to truth and concluded the model was
over-smoothing. That was wrong — a calibrated ensemble mean is legitimately smoother than
any single field, so a falling ratio was expected, not a defect.

Corrected, it reverses: members carry **too much** small-scale power, because the
precipitation sampler draws every grid point independently.

In [ ]:
fig, ax = panel("Small-scale power, single member", "power ratio (pred / truth)")
add_spread(ax, "stage4", "member_spec")
add_spread(ax, "r2",     "member_spec")
add_ref(ax, 1.0, "truth")
finish(ax, xmax=10, legend="center right", xtick=2)
save(fig, "fig7_member_power"); plt.show()

## Fig 8 — extremes in a single forecast

Round 2's individual forecasts under-represent the 99th percentile by ~19% at day 10.
This is what matters for rare-event sampling, where AI-RES draws trajectories and reads
statistics off them.

In [ ]:
fig, ax = panel("p99 of daily precipitation, single member", "p99 (mm/day)")
add_spread(ax, "stage4", "q99_obs", **STYLE["obs"])
add_spread(ax, "stage4", "q99_member")
add_spread(ax, "r2",     "q99_member")
finish(ax, xmax=10, legend="lower left", xtick=2)
save(fig, "fig8_member_p99"); plt.show()

## Fig 9 — ensemble dispersion

Spread/skill should be ~1.0. Both models sit at 0.33–0.44, and Round 2's longer rollouts
made it *worse*. This is the largest unfixed gap and the main target of Round 3.

In [ ]:
fig, ax = panel("Ensemble dispersion", "spread / skill")
add_spread(ax, "stage4", "spread_skill")
add_spread(ax, "r2",     "spread_skill")
add_ref(ax, 1.0, "calibrated")
ax.set_ylim(0, 1.15)
finish(ax, xmax=10, legend="center right", xtick=2)
save(fig, "fig9_spread_skill"); plt.show()

## Build your own

Template — copy, edit the `add(...)` lines, rename, run.

In [ ]:
fig, ax = panel("My figure", "y label")
add(ax, "v11", "pr_crps_global")
add(ax, "r2",  "pr_crps_global")
# add(ax, "stage4", "pr_crps_global", color="#009E73", label="custom label")
finish(ax, xmax=15)
# save(fig, "my_figure")
plt.show()

## Numbers for slide text

Exact values to paste, rather than reading them off a chart.

In [ ]:
def at(xs, ys, x):
    return ys[int(np.argmin(np.abs(np.array(xs) - x)))]

print("Global RMSE at day 1 / 5 / 15")
print(f"{'model':9s} {'t2m (K)':>21s}   {'precip (mm/day)':>23s}")
for k in ["v11", "stage1", "stage2", "stage3", "stage4", "r2"]:
    t = [at(LEAD, M[k]["tas_rmse_global"], d) for d in (1, 5, 15)]
    p = [at(LEAD, M[k]["pr_rmse_global"], d) for d in (1, 5, 15)]
    print(f"{k:9s} " + " ".join(f"{v:6.3f}" for v in t) + "  " + " ".join(f"{v:7.2f}" for v in p))

print("\nDaily-mean precip CRPS (mm/day)")
for d in (1, 3, 5, 7, 10):
    a = at(DAYS, DIAG["stage4"]["crps_pred"], d); b = at(DAYS, DIAG["r2"]["crps_pred"], d)
    print(f"  day {d:2d}   stage4 {a:.3f}   round2 {b:.3f}   {100*(b-a)/a:+.0f}%")

print("\nRound 3 targets at day 10")
for k in ("stage4", "r2"):
    s = SPREAD[k]
    print(f"  {k:7s} member power {s['member_spec'][-1]:.2f} (want 1.0) | "
          f"p99 {s['q99_member'][-1]:.1f} vs obs {s['q99_obs'][-1]:.1f} | "
          f"spread/skill {s['spread_skill'][-1]:.2f} (want ~1.0)")

print("\nfigures on disk:")
for f in sorted(glob.glob(f"{OUT}/*.png")):
    print("  ", f)